In [1]:
"""
Enhanced BERT from-scratch + downstream fine-tuning with comprehensive plotting
Includes larger datasets and detailed evaluation metrics
"""
import os
import random
import math
import time
import json
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from datasets import load_dataset
from transformers import BertTokenizerFast, AutoModelForSequenceClassification, AutoModelForQuestionAnswering
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, roc_curve, auc
import seaborn as sns
from tqdm.auto import tqdm
import pandas as pd

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ---------------------------
# Config / Hyperparameters
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


# Model config
MODEL_DIM = 256
NUM_HEADS = 4
NUM_LAYERS = 4
FF_DIM = 1024
MAX_LEN = 128
VOCAB_SIZE = None

BATCH_SIZE_MLM = 32
BATCH_SIZE_FT = 32
MLM_EPOCHS = 3
FT_EPOCHS = 4

MASK_PROB = 0.15
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# INCREASED DATASET SIZES for better accuracy
MLM_EXAMPLES = 50000      # Increased from 20000
SST2_EXAMPLES = 20000     # Increased from 4000
SQUAD_EXAMPLES = 5000     # Increased from 1000
SNLI_EXAMPLES = 20000     # Increased from 4000

# ---------------------------
# Tokenizer
# ---------------------------
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id

print(f"Vocab size: {VOCAB_SIZE}, pad id: {pad_token_id}")

Device: cuda
Vocab size: 30522, pad id: 0


In [2]:


# ---------------------------
# Helper utilities
# ---------------------------
def collate_mlm(batch_inputs: List[List[int]], pad_id=0, max_len=MAX_LEN):
    input_ids = [ids[:max_len] for ids in batch_inputs]
    maxl = max(len(x) for x in input_ids)
    padded = [x + [pad_id] * (maxl - len(x)) for x in input_ids]
    input_ids = torch.tensor(padded, dtype=torch.long)
    return input_ids

def mask_tokens(inputs: torch.Tensor, tokenizer, mask_prob=MASK_PROB):
    labels = inputs.clone()
    probability_matrix = torch.full(labels.shape, mask_prob)
    special_tokens_mask = [
        tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) 
        for val in labels.tolist()
    ]
    special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
    probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
    padding_mask = labels.eq(tokenizer.pad_token_id)
    probability_matrix.masked_fill_(padding_mask, value=0.0)

    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100

    indices_replaced = (torch.bernoulli(torch.full(labels.shape, 0.8)).bool()) & masked_indices
    inputs[indices_replaced] = tokenizer.mask_token_id

    indices_random = (torch.bernoulli(torch.full(labels.shape, 0.5)).bool()) & masked_indices & ~indices_replaced
    random_words = torch.randint(low=0, high=tokenizer.vocab_size, size=labels.shape, dtype=torch.long)
    inputs[indices_random] = random_words[indices_random]

    return inputs, labels

# ---------------------------
# Build small BERT-like encoder
# ---------------------------
class SmallBERT(nn.Module):
    def __init__(self, vocab_size, model_dim=MODEL_DIM, num_layers=NUM_LAYERS, 
                 num_heads=NUM_HEADS, ff_dim=FF_DIM, max_len=MAX_LEN):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, model_dim)
        self.pos_emb = nn.Embedding(max_len, model_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=num_heads, dim_feedforward=ff_dim, 
            activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layernorm = nn.LayerNorm(model_dim)
        self.model_dim = model_dim
        self.mlm_head = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.GELU(),
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, vocab_size)
        )

    def forward(self, input_ids, attention_mask=None):
        b, l = input_ids.shape
        pos = torch.arange(0, l, device=input_ids.device).unsqueeze(0).expand(b, -1)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        x = self.layernorm(x)
        
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0
        else:
            key_padding_mask = None
            
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        logits = self.mlm_head(x)
        return logits, x

# ---------------------------
# MLM Dataset
# ---------------------------
class MLMDataset(Dataset):
    def __init__(self, texts: List[str], tokenizer: BertTokenizerFast, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.inputs = []
        for t in texts:
            toks = tokenizer.encode(t, add_special_tokens=True, truncation=True, max_length=max_len)
            if len(toks) > 1:
                self.inputs.append(toks)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long)

def mlm_collate_fn(batch):
    batch_ids = [b.tolist() for b in batch]
    input_ids = collate_mlm(batch_ids, pad_id=tokenizer.pad_token_id, max_len=MAX_LEN)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    input_ids_masked, labels = mask_tokens(input_ids.clone(), tokenizer)
    return input_ids_masked.to(device), attention_mask.to(device), labels.to(device)

# ---------------------------
# Task-specific heads
# ---------------------------
class ClassificationHead(nn.Module):
    def __init__(self, encoder_dim, num_classes=2):
        super().__init__()
        self.dropout = nn.Dropout(0.1)
        self.pool = nn.Linear(encoder_dim, encoder_dim)
        self.classifier = nn.Linear(encoder_dim, num_classes)
        
    def forward(self, hidden_states, attention_mask=None):
        cls_repr = hidden_states[:, 0, :]
        cls_repr = self.dropout(cls_repr)
        x = torch.tanh(self.pool(cls_repr))
        logits = self.classifier(x)
        return logits

class QAHead(nn.Module):
    def __init__(self, encoder_dim):
        super().__init__()
        self.qa_outputs = nn.Linear(encoder_dim, 2)
        
    def forward(self, hidden_states):
        logits = self.qa_outputs(hidden_states)
        start_logits, end_logits = logits.split(1, dim=-1)
        return start_logits.squeeze(-1), end_logits.squeeze(-1)

class PairClassificationHead(nn.Module):
    """For SNLI semantic similarity"""
    def __init__(self, encoder_dim, num_classes=3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(encoder_dim * 4, encoder_dim),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(encoder_dim, num_classes)
        )
        
    def forward(self, cls1, cls2):
        # Concatenate: [cls1; cls2; |cls1-cls2|; cls1*cls2]
        merged = torch.cat([cls1, cls2, torch.abs(cls1 - cls2), cls1 * cls2], dim=1)
        return self.classifier(merged)

# ---------------------------
# Datasets
# ---------------------------
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.enc = tokenizer(texts, padding='max_length', truncation=True, max_length=max_len)
        self.labels = labels
        
    def __len__(self): 
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

class PairDataset(Dataset):
    def __init__(self, texts1, texts2, labels, tokenizer, max_len=MAX_LEN):
        self.texts1 = texts1
        self.texts2 = texts2
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self): 
        return len(self.labels)
    
    def __getitem__(self, idx):
        enc1 = self.tokenizer(self.texts1[idx], padding='max_length', 
                             truncation=True, max_length=self.max_len)
        enc2 = self.tokenizer(self.texts2[idx], padding='max_length', 
                             truncation=True, max_length=self.max_len)
        return {
            'input_ids1': torch.tensor(enc1['input_ids']),
            'attention_mask1': torch.tensor(enc1['attention_mask']),
            'input_ids2': torch.tensor(enc2['input_ids']),
            'attention_mask2': torch.tensor(enc2['attention_mask']),
            'labels': torch.tensor(self.labels[idx])
        }

class SquadDataset(Dataset):
    def __init__(self, prepared, max_len=MAX_LEN):
        self.data = prepared
        self.max_len = max_len
        
    def __len__(self): 
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        input_ids = d['input_ids'][:self.max_len] + [tokenizer.pad_token_id] * max(0, self.max_len - len(d['input_ids']))
        attention_mask = d['attention_mask'][:self.max_len] + [0] * max(0, self.max_len - len(d['attention_mask']))
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'start_positions': torch.tensor(d['start'], dtype=torch.long),
            'end_positions': torch.tensor(d['end'], dtype=torch.long)
        }

# ---------------------------
# Data preparation functions
# ---------------------------
def prepare_squad_examples(dataset, tokenizer, max_len=MAX_LEN, num_examples=SQUAD_EXAMPLES):
    inputs = []
    for i, ex in enumerate(dataset):
        if i >= num_examples:
            break
        if len(ex['answers']['text']) == 0:
            continue
            
        context = ex['context']
        question = ex['question']
        answer_text = ex['answers']['text'][0]
        answer_start = ex['answers']['answer_start'][0]
        
        enc = tokenizer(question, context, truncation='only_second', 
                       max_length=max_len, return_offsets_mapping=True)
        
        offsets = enc['offset_mapping']
        seq_ids = enc.sequence_ids()
        
        start_char = answer_start
        end_char = answer_start + len(answer_text)
        token_start, token_end = None, None
        
        for idx, (off, sid) in enumerate(zip(offsets, seq_ids)):
            if sid != 1:
                continue
            if off[0] <= start_char < off[1]:
                token_start = idx
            if off[0] < end_char <= off[1]:
                token_end = idx
                
        if token_start is not None and token_end is not None:
            inputs.append({
                'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask'],
                'start': token_start,
                'end': token_end
            })
    return inputs

# ---------------------------
# Training functions
# ---------------------------
def pretrain_mlm(model, dataloader, optimizer, epochs=MLM_EPOCHS):
    model.train()
    history = {'train_loss': [], 'perplexity': []}
    mlm_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    
    for ep in range(epochs):
        pbar = tqdm(dataloader, desc=f"MLM Ep{ep+1}/{epochs}")
        running_loss = 0.0
        n = 0
        
        for input_ids_masked, attention_mask, labels in pbar:
            optimizer.zero_grad()
            logits, _ = model(input_ids_masked, attention_mask=attention_mask)
            loss = mlm_loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            n += 1
            pbar.set_postfix({'loss': running_loss / n})
            
        epoch_loss = running_loss / max(1, n)
        perplexity = math.exp(min(epoch_loss, 10))
        
        print(f"Epoch {ep+1} MLM loss: {epoch_loss:.4f}, Perplexity: {perplexity:.2f}")
        history['train_loss'].append(epoch_loss)
        history['perplexity'].append(perplexity)
        
    return history

def train_classification_scratch(encoder_model, classifier_head, train_loader, val_loader, 
                                 epochs=FT_EPOCHS, lr=2e-5):
    encoder_model.to(device)
    classifier_head.to(device)
    optimizer = optim.AdamW(list(encoder_model.parameters()) + 
                           list(classifier_head.parameters()), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': []
    }
    
    for ep in range(epochs):
        # Training
        encoder_model.train()
        classifier_head.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch in tqdm(train_loader, desc=f"Train Ep{ep+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            _, hidden = encoder_model(input_ids, attention_mask=attention_mask)
            cls_logits = classifier_head(hidden, attention_mask)
            loss = loss_fn(cls_logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(encoder_model.parameters()) + 
                                          list(classifier_head.parameters()), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            pred = torch.argmax(cls_logits, dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        
        # Validation
        val_metrics = evaluate_classification(encoder_model, classifier_head, val_loader)
        
        print(f"Epoch {ep+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['accuracy']:.4f}, "
              f"val_f1={val_metrics['f1']:.4f}")
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        
    return history

def evaluate_classification(encoder_model, classifier_head, data_loader):
    encoder_model.eval()
    classifier_head.eval()
    
    all_preds = []
    all_labels = []
    loss_sum = 0.0
    loss_fn = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            _, hidden = encoder_model(input_ids, attention_mask=attention_mask)
            cls_logits = classifier_head(hidden, attention_mask)
            loss = loss_fn(cls_logits, labels)
            
            pred = torch.argmax(cls_logits, dim=1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            loss_sum += loss.item() * labels.size(0)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )
    
    return {
        'loss': loss_sum / len(all_labels),
        'accuracy': np.mean(np.array(all_preds) == np.array(all_labels)),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

def train_pair_classification_scratch(encoder_model, pair_head, train_loader, val_loader,
                                      epochs=FT_EPOCHS, lr=2e-5):
    encoder_model.to(device)
    pair_head.to(device)
    optimizer = optim.AdamW(list(encoder_model.parameters()) + 
                           list(pair_head.parameters()), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    for ep in range(epochs):
        # Training
        encoder_model.train()
        pair_head.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch in tqdm(train_loader, desc=f"Train SNLI Ep{ep+1}/{epochs}"):
            input_ids1 = batch['input_ids1'].to(device)
            attention_mask1 = batch['attention_mask1'].to(device)
            input_ids2 = batch['input_ids2'].to(device)
            attention_mask2 = batch['attention_mask2'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            _, hidden1 = encoder_model(input_ids1, attention_mask=attention_mask1)
            _, hidden2 = encoder_model(input_ids2, attention_mask=attention_mask2)
            
            cls1 = hidden1[:, 0, :]
            cls2 = hidden2[:, 0, :]
            
            logits = pair_head(cls1, cls2)
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(encoder_model.parameters()) + 
                                          list(pair_head.parameters()), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            pred = torch.argmax(logits, dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        
        # Validation
        val_metrics = evaluate_pair_classification(encoder_model, pair_head, val_loader)
        
        print(f"Epoch {ep+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['accuracy']:.4f}")
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['f1'])
        
    return history

def evaluate_pair_classification(encoder_model, pair_head, data_loader):
    encoder_model.eval()
    pair_head.eval()
    
    all_preds = []
    all_labels = []
    loss_sum = 0.0
    loss_fn = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids1 = batch['input_ids1'].to(device)
            attention_mask1 = batch['attention_mask1'].to(device)
            input_ids2 = batch['input_ids2'].to(device)
            attention_mask2 = batch['attention_mask2'].to(device)
            labels = batch['labels'].to(device)
            
            _, hidden1 = encoder_model(input_ids1, attention_mask=attention_mask1)
            _, hidden2 = encoder_model(input_ids2, attention_mask=attention_mask2)
            
            cls1 = hidden1[:, 0, :]
            cls2 = hidden2[:, 0, :]
            
            logits = pair_head(cls1, cls2)
            loss = loss_fn(logits, labels)
            
            pred = torch.argmax(logits, dim=1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            loss_sum += loss.item() * labels.size(0)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    
    return {
        'loss': loss_sum / len(all_labels),
        'accuracy': np.mean(np.array(all_preds) == np.array(all_labels)),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

def train_qa_scratch(encoder_model, qa_head, train_loader, val_loader=None,
                    epochs=FT_EPOCHS, lr=2e-5):
    encoder_model.to(device)
    qa_head.to(device)
    optimizer = optim.AdamW(list(encoder_model.parameters()) + 
                           list(qa_head.parameters()), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_loss': []}
    
    for ep in range(epochs):
        # Training
        encoder_model.train()
        qa_head.train()
        running_loss = 0.0
        
        for batch in tqdm(train_loader, desc=f"Train QA Ep{ep+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            start_positions = batch['start_positions'].to(device)
            end_positions = batch['end_positions'].to(device)
            
            optimizer.zero_grad()
            _, hidden = encoder_model(input_ids, attention_mask=attention_mask)
            start_logits, end_logits = qa_head(hidden)
            
            loss_start = loss_fn(start_logits, start_positions)
            loss_end = loss_fn(end_logits, end_positions)
            loss = (loss_start + loss_end) / 2.0
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(encoder_model.parameters()) + 
                                          list(qa_head.parameters()), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            
        train_loss = running_loss / len(train_loader)
        
        # Validation if available
        if val_loader:
            val_loss = evaluate_qa(encoder_model, qa_head, val_loader)
            print(f"QA Epoch {ep+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
            history['val_loss'].append(val_loss)
        else:
            print(f"QA Epoch {ep+1}: train_loss={train_loss:.4f}")
            
        history['train_loss'].append(train_loss)
        
    return history

def evaluate_qa(encoder_model, qa_head, data_loader):
    encoder_model.eval()
    qa_head.eval()
    
    loss_sum = 0.0
    loss_fn = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            start_positions = batch['start_positions'].to(device)
            end_positions = batch['end_positions'].to(device)
            
            _, hidden = encoder_model(input_ids, attention_mask=attention_mask)
            start_logits, end_logits = qa_head(hidden)
            
            loss_start = loss_fn(start_logits, start_positions)
            loss_end = loss_fn(end_logits, end_positions)
            loss = (loss_start + loss_end) / 2.0
            
            loss_sum += loss.item()
            
    return loss_sum / len(data_loader)

def finetune_hf_classification(pretrained_model_name, train_loader, val_loader,
                               num_labels=2, epochs=FT_EPOCHS, lr=2e-5):
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name, num_labels=num_labels
    ).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    for ep in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch in tqdm(train_loader, desc=f"HF Train Ep{ep+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        
        # Validation
        model.eval()
        all_preds = []
        all_labels = []
        val_loss_sum = 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                preds = torch.argmax(outputs.logits, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                val_loss_sum += outputs.loss.item() * labels.size(0)
        
        val_loss = val_loss_sum / len(all_labels)
        val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        _, _, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='macro' if num_labels > 2 else 'binary', zero_division=0
        )
        
        print(f"HF Epoch {ep+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, val_f1={f1:.4f}")
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(f1)
        
    return history, model

def finetune_hf_qa(pretrained_model_name, train_loader, val_loader=None, 
                   epochs=FT_EPOCHS, lr=3e-5):
    model = AutoModelForQuestionAnswering.from_pretrained(pretrained_model_name).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    history = {'train_loss': [], 'val_loss': []}
    
    for ep in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        
        for batch in tqdm(train_loader, desc=f"HF QA Ep{ep+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            start_positions = batch['start_positions'].to(device)
            end_positions = batch['end_positions'].to(device)
            
            optimizer.zero_grad()
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                start_positions=start_positions,
                end_positions=end_positions
            )
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            
        train_loss = running_loss / len(train_loader)
        
        # Validation if available
        if val_loader:
            model.eval()
            val_loss_sum = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    start_positions = batch['start_positions'].to(device)
                    end_positions = batch['end_positions'].to(device)
                    
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        start_positions=start_positions,
                        end_positions=end_positions
                    )
                    val_loss_sum += outputs.loss.item()
                    
            val_loss = val_loss_sum / len(val_loader)
            print(f"HF QA Epoch {ep+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
            history['val_loss'].append(val_loss)
        else:
            print(f"HF QA Epoch {ep+1}: train_loss={train_loss:.4f}")
            
        history['train_loss'].append(train_loss)
        
    return history, model

# ---------------------------
# COMPREHENSIVE PLOTTING FUNCTIONS
# ---------------------------

def plot_history(history, title, save_path=None):
    """Plot training history with multiple metrics"""
    metrics = [k for k in history.keys() if k not in ['predictions', 'labels']]
    n_metrics = len(metrics)
    
    if n_metrics == 0:
        return
        
    fig, axes = plt.subplots(1, min(n_metrics, 3), figsize=(15, 4))
    if n_metrics == 1:
        axes = [axes]
    elif n_metrics == 2:
        axes = [axes[0], axes[1]]
    
    for idx, metric in enumerate(metrics[:3]):
        ax = axes[idx] if n_metrics > 1 else axes[0]
        ax.plot(history[metric], marker='o', linewidth=2, markersize=6)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
        ax.set_title(f"{title} - {metric.replace('_', ' ').title()}", fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_model_comparison(results_dict):
    """Compare multiple models across metrics"""
    # Extract metrics
    models = list(results_dict.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, metric in enumerate(metrics):
        values = [results_dict[m].get(metric, 0) for m in models]
        
        bars = axes[idx].bar(range(len(models)), values, color=plt.cm.Set3(range(len(models))))
        axes[idx].set_xticks(range(len(models)))
        axes[idx].set_xticklabels(models, rotation=45, ha='right')
        axes[idx].set_ylabel(metric.title(), fontsize=12)
        axes[idx].set_title(f"{metric.title()} Comparison", fontsize=13, fontweight='bold')
        axes[idx].set_ylim([0, 1.0])
        axes[idx].grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for bar, val in zip(bars, values):
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                          f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix(y_true, y_pred, labels, title, save_path=None):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Count'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_training_comparison(scratch_history, hf_history, task_name):
    """Compare scratch vs HF training curves"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Training Loss
    axes[0].plot(scratch_history['train_loss'], marker='o', label='Scratch BERT', linewidth=2)
    axes[0].plot(hf_history['train_loss'], marker='s', label='HF BERT', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Training Loss', fontsize=12)
    axes[0].set_title(f'{task_name} - Training Loss', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Validation Loss
    if 'val_loss' in scratch_history and 'val_loss' in hf_history:
        axes[1].plot(scratch_history['val_loss'], marker='o', label='Scratch BERT', linewidth=2)
        axes[1].plot(hf_history['val_loss'], marker='s', label='HF BERT', linewidth=2)
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('Validation Loss', fontsize=12)
        axes[1].set_title(f'{task_name} - Validation Loss', fontsize=13, fontweight='bold')
        axes[1].legend(fontsize=11)
        axes[1].grid(True, alpha=0.3)
    
    # Validation Accuracy
    if 'val_acc' in scratch_history and 'val_acc' in hf_history:
        axes[2].plot(scratch_history['val_acc'], marker='o', label='Scratch BERT', linewidth=2)
        axes[2].plot(hf_history['val_acc'], marker='s', label='HF BERT', linewidth=2)
        axes[2].set_xlabel('Epoch', fontsize=12)
        axes[2].set_ylabel('Validation Accuracy', fontsize=12)
        axes[2].set_title(f'{task_name} - Validation Accuracy', fontsize=13, fontweight='bold')
        axes[2].legend(fontsize=11)
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{task_name.lower().replace(" ", "_")}_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_metrics_over_epochs(scratch_history, hf_history, task_name):
    """Plot multiple metrics over epochs"""
    metrics = ['train_acc', 'val_acc', 'val_precision', 'val_recall', 'val_f1']
    available_metrics = [m for m in metrics if m in scratch_history and m in hf_history]
    
    if not available_metrics:
        return
        
    n_metrics = len(available_metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5*n_metrics, 4))
    
    if n_metrics == 1:
        axes = [axes]
    
    for idx, metric in enumerate(available_metrics):
        axes[idx].plot(scratch_history[metric], marker='o', label='Scratch BERT', linewidth=2)
        axes[idx].plot(hf_history[metric], marker='s', label='HF BERT', linewidth=2)
        axes[idx].set_xlabel('Epoch', fontsize=12)
        axes[idx].set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
        axes[idx].set_title(f"{metric.replace('_', ' ').title()}", fontsize=13, fontweight='bold')
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3)
    
    plt.suptitle(f'{task_name} - Detailed Metrics', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{task_name.lower().replace(" ", "_")}_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()

def create_summary_table(results_dict):
    """Create a summary table of all results"""
    df = pd.DataFrame(results_dict).T
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS SUMMARY")
    print("="*80)
    print(df.to_string())
    print("="*80 + "\n")
    
    # Save to CSV
    df.to_csv('results_summary.csv')
    print("Results saved to 'results_summary.csv'")
    
    return df



In [3]:
# ---------------------------
# MAIN EXECUTION
# ---------------------------

def main():
    print("\n" + "="*80)
    print("BERT FROM SCRATCH - COMPREHENSIVE TRAINING AND EVALUATION")
    print("="*80 + "\n")
    
    # Load and prepare MLM data
    print("Loading pretraining corpus (WikiText-2)...")
    wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    texts = [x.strip() for x in wikitext["text"][:MLM_EXAMPLES] if len(x.strip()) > 0]
    print(f"MLM sentences loaded: {len(texts)}")
    
    mlm_ds = MLMDataset(texts, tokenizer, max_len=MAX_LEN)
    mlm_loader = DataLoader(mlm_ds, batch_size=BATCH_SIZE_MLM, shuffle=True, collate_fn=mlm_collate_fn)
    
    # Initialize and pretrain BERT
    print("\n" + "="*80)
    print("STEP 1: PRETRAINING BERT WITH MASKED LANGUAGE MODELING")
    print("="*80)
    bert_scratch = SmallBERT(vocab_size=VOCAB_SIZE).to(device)
    optim_mlm = optim.AdamW(bert_scratch.parameters(), lr=5e-4, weight_decay=0.01)
    
    mlm_history = pretrain_mlm(bert_scratch, mlm_loader, optim_mlm, epochs=MLM_EPOCHS)
    
    # Plot MLM results
    plot_history(mlm_history, "Masked Language Modeling Pretraining", "mlm_training.png")
    
    # ========== TASK 1: SENTIMENT CLASSIFICATION (SST-2) ==========
    print("\n" + "="*80)
    print("STEP 2: SENTIMENT CLASSIFICATION (SST-2)")
    print("="*80)
    
    print("Loading SST-2 dataset...")
    sst2 = load_dataset("glue", "sst2")
    train_sst = sst2["train"]["sentence"][:SST2_EXAMPLES]
    train_sst_labels = sst2["train"]["label"][:SST2_EXAMPLES]
    val_sst = sst2["validation"]["sentence"][:2000]  # Increased validation set
    val_sst_labels = sst2["validation"]["label"][:2000]
    
    print(f"Training samples: {len(train_sst)}, Validation samples: {len(val_sst)}")
    
    train_sst_ds = SentimentDataset(train_sst, train_sst_labels, tokenizer)
    val_sst_ds = SentimentDataset(val_sst, val_sst_labels, tokenizer)
    train_sst_loader = DataLoader(train_sst_ds, batch_size=BATCH_SIZE_FT, shuffle=True)
    val_sst_loader = DataLoader(val_sst_ds, batch_size=BATCH_SIZE_FT, shuffle=False)
    
    # Train scratch BERT
    print("\nTraining Scratch BERT on SST-2...")
    sst_head = ClassificationHead(encoder_dim=MODEL_DIM, num_classes=2)
    sst_history = train_classification_scratch(bert_scratch, sst_head, train_sst_loader, 
                                               val_sst_loader, epochs=FT_EPOCHS)
    
    # Train HF BERT
    print("\nTraining HuggingFace BERT on SST-2...")
    hf_sst_history, hf_sst_model = finetune_hf_classification(
        "bert-base-uncased", train_sst_loader, val_sst_loader, 
        num_labels=2, epochs=FT_EPOCHS
    )
    
    # Evaluate both models
    print("\nEvaluating models on SST-2...")
    sst_eval_scratch = evaluate_classification(bert_scratch, sst_head, val_sst_loader)
    
    hf_sst_model.eval()
    hf_preds = []
    hf_labels = []
    with torch.no_grad():
        for batch in val_sst_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = hf_sst_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            hf_preds.extend(preds.cpu().numpy())
            hf_labels.extend(labels.cpu().numpy())
    
    p, r, f, _ = precision_recall_fscore_support(hf_labels, hf_preds, average='binary', zero_division=0)
    sst_eval_hf = {
        'accuracy': np.mean(np.array(hf_preds) == np.array(hf_labels)),
        'precision': p, 'recall': r, 'f1': f,
        'predictions': hf_preds, 'labels': hf_labels
    }
    
    # Plot SST-2 results
    plot_training_comparison(sst_history, hf_sst_history, "SST-2 Sentiment")
    plot_metrics_over_epochs(sst_history, hf_sst_history, "SST-2 Sentiment")
    plot_confusion_matrix(sst_eval_scratch['labels'], sst_eval_scratch['predictions'],
                         ['Negative', 'Positive'], "Scratch BERT - SST-2", "sst2_scratch_cm.png")
    plot_confusion_matrix(sst_eval_hf['labels'], sst_eval_hf['predictions'],
                         ['Negative', 'Positive'], "HF BERT - SST-2", "sst2_hf_cm.png")
    
    # ========== TASK 2: SEMANTIC SIMILARITY (SNLI) ==========
    print("\n" + "="*80)
    print("STEP 3: SEMANTIC SIMILARITY (SNLI)")
    print("="*80)
    
    print("Loading SNLI dataset...")
    snli = load_dataset("snli")
    
    # Filter valid labels
    train_data = [(p, h, l) for p, h, l in zip(
        snli['train']['premise'][:SNLI_EXAMPLES],
        snli['train']['hypothesis'][:SNLI_EXAMPLES],
        snli['train']['label'][:SNLI_EXAMPLES]
    ) if l != -1]
    
    val_data = [(p, h, l) for p, h, l in zip(
        snli['validation']['premise'][:5000],
        snli['validation']['hypothesis'][:5000],
        snli['validation']['label'][:5000]
    ) if l != -1]
    
    train_premises, train_hypotheses, train_snli_labels = zip(*train_data)
    val_premises, val_hypotheses, val_snli_labels = zip(*val_data)
    
    print(f"Training samples: {len(train_snli_labels)}, Validation samples: {len(val_snli_labels)}")
    
    train_snli_ds = PairDataset(train_premises, train_hypotheses, train_snli_labels, tokenizer)
    val_snli_ds = PairDataset(val_premises, val_hypotheses, val_snli_labels, tokenizer)
    train_snli_loader = DataLoader(train_snli_ds, batch_size=BATCH_SIZE_FT, shuffle=True)
    val_snli_loader = DataLoader(val_snli_ds, batch_size=BATCH_SIZE_FT, shuffle=False)
    
    # Train scratch BERT
    print("\nTraining Scratch BERT on SNLI...")
    snli_head = PairClassificationHead(encoder_dim=MODEL_DIM, num_classes=3)
    snli_history = train_pair_classification_scratch(bert_scratch, snli_head, 
                                                     train_snli_loader, val_snli_loader, epochs=FT_EPOCHS)
    
    # Train HF BERT (using sequence classification with pair inputs)
    print("\nTraining HuggingFace BERT on SNLI...")
    # Create combined dataset for HF model
    train_combined = [f"{p} [SEP] {h}" for p, h in zip(train_premises, train_hypotheses)]
    val_combined = [f"{p} [SEP] {h}" for p, h in zip(val_premises, val_hypotheses)]
    
    train_snli_hf_ds = SentimentDataset(train_combined, train_snli_labels, tokenizer)
    val_snli_hf_ds = SentimentDataset(val_combined, val_snli_labels, tokenizer)
    train_snli_hf_loader = DataLoader(train_snli_hf_ds, batch_size=BATCH_SIZE_FT, shuffle=True)
    val_snli_hf_loader = DataLoader(val_snli_hf_ds, batch_size=BATCH_SIZE_FT, shuffle=False)
    
    hf_snli_history, hf_snli_model = finetune_hf_classification(
        "bert-base-uncased", train_snli_hf_loader, val_snli_hf_loader,
        num_labels=3, epochs=FT_EPOCHS
    )
    
    # Evaluate both models
    print("\nEvaluating models on SNLI...")
    snli_eval_scratch = evaluate_pair_classification(bert_scratch, snli_head, val_snli_loader)
    
    hf_snli_model.eval()
    hf_snli_preds = []
    hf_snli_labels = []
    with torch.no_grad():
        for batch in val_snli_hf_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = hf_snli_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            hf_snli_preds.extend(preds.cpu().numpy())
            hf_snli_labels.extend(labels.cpu().numpy())
    
    p, r, f, _ = precision_recall_fscore_support(hf_snli_labels, hf_snli_preds, average='macro', zero_division=0)
    snli_eval_hf = {
        'accuracy': np.mean(np.array(hf_snli_preds) == np.array(hf_snli_labels)),
        'precision': p, 'recall': r, 'f1': f,
        'predictions': hf_snli_preds, 'labels': hf_snli_labels
    }
    
    # Plot SNLI results
    plot_training_comparison(snli_history, hf_snli_history, "SNLI Semantic Similarity")
    plot_metrics_over_epochs(snli_history, hf_snli_history, "SNLI Semantic Similarity")
    plot_confusion_matrix(snli_eval_scratch['labels'], snli_eval_scratch['predictions'],
                         ['Entailment', 'Neutral', 'Contradiction'], 
                         "Scratch BERT - SNLI", "snli_scratch_cm.png")
    plot_confusion_matrix(snli_eval_hf['labels'], snli_eval_hf['predictions'],
                         ['Entailment', 'Neutral', 'Contradiction'],
                         "HF BERT - SNLI", "snli_hf_cm.png")
    
    # ========== TASK 3: QUESTION ANSWERING (SQuAD) ==========
    print("\n" + "="*80)
    print("STEP 4: QUESTION ANSWERING (SQuAD)")
    print("="*80)
    
    print("Loading SQuAD dataset...")
    squad = load_dataset("squad")
    squad_train = prepare_squad_examples(squad['train'], tokenizer, num_examples=SQUAD_EXAMPLES)
    squad_val = prepare_squad_examples(squad['validation'], tokenizer, num_examples=1000)
    
    print(f"Training samples: {len(squad_train)}, Validation samples: {len(squad_val)}")
    
    squad_train_ds = SquadDataset(squad_train)
    squad_val_ds = SquadDataset(squad_val)
    squad_train_loader = DataLoader(squad_train_ds, batch_size=BATCH_SIZE_FT, shuffle=True)
    squad_val_loader = DataLoader(squad_val_ds, batch_size=BATCH_SIZE_FT, shuffle=False)
    
    # Train scratch BERT
    print("\nTraining Scratch BERT on SQuAD...")
    qa_head = QAHead(encoder_dim=MODEL_DIM)
    squad_history = train_qa_scratch(bert_scratch, qa_head, squad_train_loader, 
                                    squad_val_loader, epochs=FT_EPOCHS)
    
    # Train HF BERT
    print("\nTraining HuggingFace BERT on SQuAD...")
    hf_squad_history, hf_squad_model = finetune_hf_qa(
        "bert-base-uncased", squad_train_loader, squad_val_loader, epochs=FT_EPOCHS
    )
    
    # Plot QA results
    plot_training_comparison(squad_history, hf_squad_history, "SQuAD Question Answering")
    
    # ========== FINAL COMPARISON ==========
    print("\n" + "="*80)
    print("FINAL COMPREHENSIVE COMPARISON")
    print("="*80 + "\n")
    
    results = {
        "Scratch-BERT-SST2": sst_eval_scratch,
        "HF-BERT-SST2": sst_eval_hf,
        "Scratch-BERT-SNLI": snli_eval_scratch,
        "HF-BERT-SNLI": snli_eval_hf
    }
    
    # Create comprehensive comparison plots
    plot_model_comparison(results)
    
    # Create summary table
    summary_df = create_summary_table(results)
    
    # Save models
    print("\nSaving models...")
    torch.save({
        'bert': bert_scratch.state_dict(),
        'sst_head': sst_head.state_dict(),
        'snli_head': snli_head.state_dict(),
        'qa_head': qa_head.state_dict()
    }, "scratch_bert_all_tasks.pth")
    
    hf_sst_model.save_pretrained("hf_bert_sst2")
    hf_snli_model.save_pretrained("hf_bert_snli")
    hf_squad_model.save_pretrained("hf_bert_squad")
    
    print("\n" + "="*80)
    print("TRAINING AND EVALUATION COMPLETE!")
    print("="*80)
    print("\nGenerated files:")
    print("- mlm_training.png: MLM pretraining curves")
    print("- sst2_sentiment_comparison.png: SST-2 training comparison")
    print("- sst2_sentiment_metrics.png: SST-2 detailed metrics")
    print("- sst2_scratch_cm.png & sst2_hf_cm.png: Confusion matrices")
    print("- snli_semantic_similarity_comparison.png: SNLI training comparison")
    print("- snli_semantic_similarity_metrics.png: SNLI detailed metrics")
    print("- snli_scratch_cm.png & snli_hf_cm.png: Confusion matrices")
    print("- squad_question_answering_comparison.png: SQuAD training comparison")
    print("- model_comparison.png: Overall model comparison")
    print("- results_summary.csv: Complete results table")
    print("="*80 + "\n")

if __name__ == "__main__":
    main()


BERT FROM SCRATCH - COMPREHENSIVE TRAINING AND EVALUATION

Loading pretraining corpus (WikiText-2)...


KeyboardInterrupt: 